# OpTC Host ML — RF, shared full-data shards

This notebook reads the same persisted 75/10/15 split and vocabulary as the other models.
Every event is retained. Run each model in a separate notebook. No local training or automatic sampling.

Set MANIFEST and VOLUME_ROOTS after attaching the output. Each volume may be mounted under a different Kaggle dataset directory.
Upload/quota decisions depend on the measured final size. Dataset capacity is not notebook RAM.


Standard RF/LightGBM fitting still needs a full training representation in RAM. The budget check deliberately stops before allocation if this runtime is too small. This is a full-data template, not a shard-by-shard incremental approximation.

In [ ]:
from pathlib import Path
import sys, json
MANIFEST = Path('/kaggle/input/optc-host-input/manifest.json')
# Keys must match manifest volumes; point each to its copied volume root.
VOLUME_ROOTS = {
    'ssd': '/kaggle/input/optc-host-input/volumes/ssd',
    'hdd': '/kaggle/input/optc-host-hdd',
    'system': '/kaggle/input/optc-host-system',
}
sys.path.insert(0, str(MANIFEST.parent))
from host_shards import HostShardDataset
from shard_metrics import evaluate_stream
data = HostShardDataset(MANIFEST, VOLUME_ROOTS)
print(json.dumps(data.meta['splits'], indent=2))
print('Full train CSR lower bound GiB:', data.csr_bytes('train') / 2**30)
OUTPUT = Path('/kaggle/working')
OUTPUT.mkdir(exist_ok=True)


In [ ]:
MODEL_NAME = 'rf'
import pickle
from sklearn.ensemble import RandomForestClassifier
# Array budget only; RF needs additional working memory. With the current
# billions of rows, a standard Kaggle notebook is expected to stop here.
# Raising this number without additional RAM does not make the full fit feasible.
X_train, y_train = data.materialize('train', max_bytes=12 * 2**30)
model = RandomForestClassifier(n_estimators=500, max_depth=20,
    min_samples_leaf=5, max_features='sqrt', class_weight='balanced',
    random_state=1, n_jobs=4)
model.fit(X_train, y_train)
del X_train, y_train
predict = lambda X: model.predict_proba(X)[:, 1]
with (OUTPUT / 'rf_model.pkl').open('wb') as handle:
    pickle.dump(model, handle)


In [ ]:
validation = evaluate_stream(data, 'val', predict)
test = evaluate_stream(data, 'test', predict, threshold=validation['threshold'])
report = {'model': MODEL_NAME, 'split_sha256':data.meta['split_sha256'],
    'vocabulary_sha256':data.meta['vocabulary_sha256'], 'seed':1,
    'validation':validation, 'test':test}
(OUTPUT / (MODEL_NAME + '_metrics.json')).write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
